# Public Portfolio Note

**Purpose:** Document preprocessing, missing-data handling, feature engineering, EDA, nonparametric tests, and aggregate figure generation for the Hospital Rating analysis.

**Inputs:** Excluded merged analytic tables and staged SDOH/POS subsets produced by the upstream workflow.

**Outputs:** Excluded cleaned modeling dataset plus aggregate figures and summary statistics.

**Public Data Limitation:** Row-level hospital previews are removed; public-facing outputs should remain aggregate or sanitized only.


# Section 5: Exploratory Data Analysis (EDA) & Feature Engineering

This section documents the preprocessing and feature-engineering workflow that turns the merged hospital dataset into the final modeling table. It also produces aggregate EDA figures and nonparametric alignment checks.

**Key Steps:**
1. Apply analytic exclusions and inspect missingness patterns.
2. Construct Kaiser / integrated HMO and academic medical center indicators.
3. Merge county-level SDOH and institutional staging fields.
4. Transform review count and standardize modeling covariates.
5. Construct percentile-rank gap outcomes for regression.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

import openpyxl
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
import math

from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# Public path constants. These row-level intermediate/modeling files are excluded from the public repository.
MERGED_HOSPITAL_DATASET_PATH = "data/excluded/merged_hospital_dataset.csv"
CA_COUNTY_SDOH_SUBSET_PATH = "data/excluded/ca_county_sdoh_subset.csv"
CMS_POS_TEACHING_INDICATORS_PATH = "data/excluded/cms_pos_teaching_indicators.csv"
CLEANED_ANALYSIS_DATASET_PATH = "data/excluded/cleaned_hospital_analysis_data.csv"
FINAL_ANALYTIC_DATASET_PATH = "data/excluded/final_analytic_dataset.csv"


In [ ]:

# Public GitHub output paths for aggregate artifacts.
PROJECT_ROOT = Path("..")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


## 5.1 Load Data

In [ ]:
df_final = pd.read_csv(MERGED_HOSPITAL_DATASET_PATH)
df_county_sdoh = pd.read_csv(CA_COUNTY_SDOH_SUBSET_PATH)
df_teaching = pd.read_csv(CMS_POS_TEACHING_INDICATORS_PATH)

df_final['cms_id'] = df_final['cms_id'].astype(str).str.split('.').str[0].str.zfill(6)
df_teaching['cms_id'] = df_teaching['cms_id'].astype(str).str.split('.').str[0].str.zfill(6)

In [ ]:
# Public copy: df.info() output removed.


## 5.2 Missing Data

In [ ]:
print('Total ', len(df_final[df_final['cms_star'].isnull()]),' cmc_star missing:')
# Public copy: row-level preview/debug output removed.
print('Total ', len(df_final[df_final['clinical_score'].isnull()]),' clinical_score missing:')
# Public copy: row-level preview/debug output removed.
print('Total ', len(df_final[df_final['patient_exp_score'].isnull()]),' patient_exp_score missing:')
# Public copy: row-level preview/debug output removed.


In [ ]:
# Missing not at random 
# new/reformed hospital and specialty not avaliable for cms evaluation
df_clean = df_final.dropna(subset = ['cms_star']).copy()
df_clean = df_clean[~df_clean['cms_name'].str.contains('RANCHO LOS AMIGOS', case=False, na=False)] # specialty
print(len(df_clean))

# Kaiser / integrated HMO institutional subgroup; descriptive indicator, not a general causal HMO variable.
df_clean['flag_is_kaiser'] = df_clean['cms_name'].str.contains('KAISER', case=False, na=False).astype(int)
# Flag for kaiser and a small emergency Hospital not suitable for cms clinical data
df_clean['flag_missing_clinical'] = df_clean['clinical_score'].isna().astype(int)
# Flag for safenet or rural area without enough survey data
df_clean['flag_missing_patient_exp'] = df_clean['patient_exp_score'].isna().astype(int)

# Public copy: df.info() output removed.
# Public copy: row-level preview/debug output removed.


In [ ]:
# missing patterns
print(f"Total {len(df_clean[df_clean['flag_is_kaiser'] == 1])} Kaiser")
kaiser_with_clinical = df_clean[(df_clean['flag_is_kaiser'] == 1) & (df_clean['flag_missing_clinical'] == 0)]

print(f"Total {len(kaiser_with_clinical)} Kaiser with clinical:")
# Public copy: row-level preview/debug output removed.

not_kaiser_missing_clinical = df_clean[(df_clean['flag_is_kaiser'] == 0) & (df_clean['flag_missing_clinical'] == 1)]
print(f"Total {len(not_kaiser_missing_clinical)} not Kaiser with missing clinical:")
# Public copy: row-level preview/debug output removed.

missing_patient_exp = df_clean[df_clean['flag_missing_patient_exp'] == 1]
print(f"Total {len(missing_patient_exp)} missing patient experience:")
# Public copy: row-level preview/debug output removed.


## 5.3 Academic Medical Center flag and county SDOH merge

In [ ]:

# merge AHRQ SDOH dataset

df_clean = df_clean.merge(
    df_county_sdoh, 
    left_on='cms_county', 
    right_on='sdoh_county', 
    how='left'
).drop(columns=['sdoh_county'])

# merge CMS dataset for seaching AMCs

df_clean = df_clean.merge(
    df_teaching[['cms_id', 'RSDNT_PHYSN_CNT', 'MDCL_SCHL_AFLTN_CD']], 
    on='cms_id', 
    how='left'
)


In [ ]:
def academic_filter(row):

    # CMS definition
    # 1 = Major Affiliation 
    # 2 = Limited Affiliation 
    # 3 = Graduate Medical Education 
    # 4 = No Affiliation 
    
    name = str(row['cms_name']).upper()
    residents = float(row.get('RSDNT_PHYSN_CNT', 0))
    code = str(row.get('MDCL_SCHL_AFLTN_CD', '4'))
    
    # Situation 1: hosital includes enough residents = teaching
    if code in ['1', '2', '3', '1.0', '2.0','3.0'] and residents > 200:
        # exclude kaiser
        if 'KAISER' not in name:
            return 1
    
    # Situation 2: hospital affiliated to medical school
    if code in ['1', '2', '3', '1.0', '2.0','3.0'] and re.search(academic_keywords, name):
        # exclude kaiser
        if 'KAISER' not in name:
            return 1
            
    return 0

# Set academic keywords for university affiliated amcs
academic_keywords = r'UNIVERSITY OF CALIFORNIA|UC|UCLA|UCSF|UCI|UCSD|UCD|USC|STANFORD|KECK|LOMA LINDA|CEDARS-SINAI'

# Flag AMCs
df_clean['flag_is_academic'] = df_clean.apply(academic_filter, axis=1)

# Exclude community hospital and a religious hospital
exclude = ['RIVERSIDE COMMUNITY HOSPITAL',
           'UCSF HEALTH ST. MARY\'S HOSPITAL']
df_clean.loc[df_clean['cms_name'].isin(exclude), 'flag_is_academic'] = 0

print(len(df_clean[df_clean['flag_is_academic'] == 1]['cms_name']))
df_acedamic = df_clean[df_clean['flag_is_academic'] == 1]
# Public copy: row-level academic-center preview removed.
# Public copy: row-level academic-center preview removed.


## 5.4 Summary Stats and Distribution 

In [ ]:
def plot_multiple_distributions(df, columns, n_cols=3, figsize_per_plot=(5, 4)):
    """
    Pipeline: EDA Distribution Density Plot
    Automatically loops through n columns and combines them into a single grid.
    """
    # 1. Calculate grid dimensions
    n_vars = len(columns)
    n_rows = math.ceil(n_vars / n_cols)
    
    # 2. Initialize the figure
    fig, axes = plt.subplots(n_rows, n_cols, 
                             figsize=(figsize_per_plot[0] * n_cols, figsize_per_plot[1] * n_rows))
    
    # Flatten the axes array to easily iterate, handle single-row edge case
    axes = axes.flatten() if n_vars > 1 else [axes]
    
    # 3. Plot Density + Histogram
    for i, col in enumerate(columns):
        if col in df.columns:
            # Dropna specifically for the column to prevent KDE errors
            clean_data = df[col].dropna()
            
            # histgram and kde
            sns.histplot(data=clean_data, kde=True, ax=axes[i], 
                         color='#2C6FAC', bins=15, stat="density", 
                         alpha=0.6, line_kws={'linewidth': 2})
            
            # Formatting 
            # axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold', pad=10)
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].grid(axis='y', linestyle='--', alpha=0.4)
            
            # Despine for a clean look
            axes[i].spines['top'].set_visible(False)
            axes[i].spines['right'].set_visible(False)
            axes[i].spines['left'].set_color('#DDDDDD')
            axes[i].spines['bottom'].set_color('#DDDDDD')
    
    # 4. Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
        
    # 5. Adjust layout and save/show
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / '01_rating_distributions.png', dpi=300)
    plt.show()

# ----- Fig.1 Distribution -----

features_to_plot = ['google_star', 'cms_star', 
                    'clinical_score','patient_exp_score',
                    'SVI_idx','Gini_idx', 'google_rating_count']

features_to_plot = ['google_star', 'google_rating_count','cms_star', 'clinical_score']
plot_multiple_distributions(df_clean, features_to_plot, n_cols=4)

In [ ]:
df_clean.describe()

## 5.5 Pre-processing

In [ ]:

# Pre-processing

# 1. log transformation for google review counts
df_clean['log_google_reviews'] = np.log(df_clean['google_rating_count'])

# 2. z-score standardiztion
scaler = StandardScaler()
cols_to_scale = ['log_google_reviews', 'SVI_idx', 'Gini_idx']

for col in cols_to_scale:
    # exclude NaN
    valid_mask = df_clean[col].notna()
    df_clean.loc[valid_mask, f'{col}_z'] = scaler.fit_transform(df_clean.loc[valid_mask, [col]])

cols_scaled = ['log_google_reviews_z', 'SVI_idx_z', 'Gini_idx_z']

# 3. percentile rank
cols_to_pct = ['google_star', 'cms_star', 'clinical_score', 'patient_exp_score']

for col in cols_to_pct:
    df_clean[f'{col}_pct'] = df_clean[col].rank(pct=True)
    # df_clean[f'{col}_pct'] = df_clean[col].rank(pct=True, method='min') # sensitivity check: consistent with using average

cols_pct = ['google_star_pct', 'cms_star_pct', 'clinical_score_pct', 'patient_exp_score_pct']

# 4. output: facility level cleaned dataset
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this section produced:
#
# - cleaned_hospital_analysis_data.csv
#   Cleaned hospital-level analysis dataset after exclusions, subgroup indicators,
#   SDOH/POS merges, transformations, and percentile-rank construction.
#
# This file is excluded from the public repository because it contains
# row-level Google Places API-derived records and merged analytic fields.
#
# Private workflow only:
# df_clean.to_csv(CLEANED_ANALYSIS_DATASET_PATH, index=False)

print("Public copy: cleaned analysis dataset CSV export disabled. Expected private output is documented above.")

# Public copy: row-level preview/debug output removed.


## 5.6 Non-parametric Methods

In [ ]:
# Spearman Correlation for raw dataset

corr_cols = ['google_star', 'cms_star', 'clinical_score','patient_exp_score', 'SVI_idx', 'Gini_idx', 'google_rating_count']
print('-'*20,'Hospital-Level Spearman Correlation:','-'*20)
corr = df_clean[corr_cols].corr(method='spearman').round(3)
display(corr['google_star'])

print('-'*20,'Spearman Correlation Heatmap:','-'*20)
# ----- Fig.2 Heatmap -----
plt.figure(figsize=(8, 6))

corr_cols = ['google_star', 'cms_star', 'clinical_score', 'patient_exp_score']
corr_labels = ['Google Star', 'CMS Star', 'CMS Clinical', 'CMS Patient Exp']
corr_matrix = df_clean[corr_cols].corr(method='spearman')

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-0.2, vmax=1, fmt=".2f",
            xticklabels=corr_labels, yticklabels=corr_labels, 
            annot_kws={"size": 14, "weight": "bold"})
# plt.title('Spearman Correlation Heatmap on Raw Data', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURE_DIR / '02_spearman_correlation_heatmap.png', dpi=300)
plt.show()

In [ ]:
# Wilcoxon Test for raw data
stat, p_val = stats.wilcoxon(df_clean['google_star'], df_clean['cms_star'])

print('-'*20,'Initial Google vs CMS','-'*20)
print(f"Wilcoxon Test P-value: {round(p_val,10)}")
print(f"Google mean: {df_clean['google_star'].mean():.2f}")
print(f"CMS mean: {df_clean['cms_star'].mean():.2f}")

## 5.7 Visualization

In [ ]:
# ----- Fig.3 Bar plot -----
def categorize_system(row):
    if row.get('flag_is_kaiser', 0) == 1:
        return 'Kaiser / integrated HMO'
    elif row.get('flag_is_academic', 0) == 1:
        return 'Academic'
    else:
        return 'Other'

df_clean['System'] = df_clean.apply(categorize_system, axis=1)

agg_df = df_clean.groupby('System')[['google_star', 'cms_star']].mean().reset_index()
agg_melt = agg_df.melt(id_vars='System', var_name='Metric', value_name='Average Star')
agg_melt['Metric'] = agg_melt['Metric'].replace({
    'google_star': 'Google Star (Crowdsourced)', 
    'cms_star': 'CMS Star (Federal Agency)'
})
system_order = ['Kaiser / integrated HMO', 'Other', 'Academic']

plt.figure(figsize=(9, 6))
ax = sns.barplot(data=agg_melt, x='System', y='Average Star', hue='Metric', 
                 palette=['#4285F4', '#DB4437'], order=system_order)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(format(p.get_height(), '.2f'), 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontweight='bold')
# plt.title('Mean divergence by institutional subgroup', fontsize=16, fontweight='bold', pad=15)
plt.ylim(0, 5)
plt.ylabel('Average Rating (1-5 Stars)')
plt.xlabel('Hospital System Type')
plt.legend(title='')
plt.tight_layout()
plt.savefig(FIGURE_DIR / '03_system_comparison_bar.png', dpi=300)
plt.show()


In [ ]:
# ----- Fig.4 Bubble Plot: Google vs. CMS Clinical pct-----
plt.figure(figsize=(10, 7))
sizes = np.clip(df_clean['google_rating_count'], 10, 3000)
sizes.name = 'Google Rating Count' 
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=2, zorder=1, label='Perfect Alignment')

sns.scatterplot(data=df_clean, x='clinical_score_pct', y='google_star_pct', 
                size=sizes, sizes=(50, 800), alpha=0.7, 
                hue='System', palette={'Kaiser / integrated HMO':'#E8562A', 'Academic':'#2C6FAC', 'Other':'#AAAAAA'}, 
                edgecolor='black', zorder=2)

# plt.title('Bubble Plot: Google vs. CMS Clinical', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('CMS Clinical Quality Percentile')
plt.ylabel('Google Rating Percentile')

legend = plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, 
                    labelspacing=1.9) 
for text in legend.get_texts():
    if text.get_text() in ['System', 'Google Rating Count']:
        text.set_fontweight('bold') 
        text.set_fontsize(11) 

plt.tight_layout()
plt.savefig(FIGURE_DIR / '04_google_cms_clinical_rank_bubble.png', dpi=300)
plt.show()

## 5.8 Gap calculation and ICC

In [ ]:
# exclude missing data for each gap analysis
# calculating gaps

df = df_clean.copy()

df['gap_star'] = df['google_star']-df['cms_star']
df['gap_star_pct'] = df['google_star_pct']-df['cms_star_pct']

df['gap_google_clinical'] = df['google_star_pct']-df['clinical_score_pct']
df['gap_google_official_exp'] = df['google_star_pct']-df['patient_exp_score_pct']

# output: gap calculated dataset for OLS
# Public copy: row-level/intermediate CSV exports are disabled.
# In the original private workflow, this section produced:
#
# - final_analytic_dataset.csv
#   Final modeling dataset with percentile-gap outcomes used by the OLS notebook.
#
# This file is excluded from the public repository because it contains
# row-level Google Places API-derived records and final modeling artifacts.
#
# Private workflow only:
# df.to_csv(FINAL_ANALYTIC_DATASET_PATH, index=False)

print("Public copy: final analytic dataset CSV export disabled. Expected private output is documented above.")


In [ ]:
# Check Intraclass Corr Coef(ICC): Considering MixedLM 
# fit null model with no predictor, only random intercept
null_model = smf.mixedlm("gap_star ~ 1", data=df, groups=df["cms_county"])
null_res = null_model.fit(reml=True)
print(null_res.summary())

# ICC
var_county = null_res.cov_re.iloc[0, 0]   # random intercept variance
var_resid  = null_res.scale               # residual variance
icc = var_county / (var_county + var_resid)
print(f"\nICC = {icc:.3f}")
print(f"Var(county) = {var_county:.4f}")
print(f"Var(resid)  = {var_resid:.4f}")